Rubtcova Vera  
Algorithms Task 6

## Goal  
The use of path search algorithms on weighted graphs (Dijkstra's, A* and Bellman
Ford algorithms)

In [ ]:
import math
import pandas as pd
import random
import matplotlib.pyplot as plt
import time

## Task 1  
- Generate a random adjacency matrix for a simple undirected weighted graph of
100 vertices and 500 edges with assigned random positive integer weights (note that
the matrix should be symmetric and contain only 0s and weights as elements). Use
Dijkstra's and Bellman-Ford algorithms to find shortest paths between a random
starting vertex and other vertices. Measure the time required to find the paths for
each algorithm.  Repeat the experiment 10 times for the same starting vertex and
calculate the average time required for the paths search of each algorithm. Analyse
the results obtained.   


In [ ]:
def generate_matrix():
  matrix = [[0] * 100 for _ in range(100)]
  edges = 0
  while edges < 500:
    i, j = random.sample(range(100), 2)
    if matrix[i][j] == 0:
      matrix[i][j] = matrix[j][i] = random.randint(1, 10)
      edges += 1
  return matrix

In [ ]:
def generate_list(matrix):
  adjacency_list = {}
  for i in range(len(matrix)):
    neighbours = []
    for j in range(len(matrix)):
      if matrix[i][j] > 0:
        neighbours.append((j, matrix[i][j]))
    adjacency_list[i] = neighbours
  return adjacency_list

In [ ]:
random.seed(200)
matrix = generate_matrix()
adj_list = generate_list(matrix)

In [ ]:
def dijkstra(start, adj_list):
  n = len(adj_list)
  dist = len(adj_list) * [999]
  dist[start] = 0
  sptSet = len(adj_list) * [False]
  for _ in range(n):
    best = float("inf")
    u = None
    for v in range(n):
      if not(sptSet[v]) and dist[v] < best:
        best = dist[v]
        u = v
    sptSet[u] = True
    for v, w in adj_list[u]:
      if dist[u] + w < dist[v]:
        dist[v] = dist[u] + w
  return dist

In [ ]:
def bellman_ford(start, adj_list):
  n = len(adj_list)
  dist = len(adj_list) * [999]
  dist[start] = 0
  for _ in range(n-1):
    for u in range(n):
      for v, w in adj_list[u]:
        if dist[u] + w < dist[v]:
          dist[v] = dist[u] + w
  return dist

In [ ]:
dot = random.randrange(100)
dist_dijkstra = []
dijkstra_times = []
for i in range(10):
  t0 = time.perf_counter()
  dist_dijkstra.append(dijkstra(dot, adj_list))
  t1 = time.perf_counter()
  dijkstra_times.append(t1 - t0)
print(dijkstra_times)

[0.0017682789994069026, 0.002648779000082868, 0.0012977160004083999, 0.000926126999729604, 0.0015311979996113223, 0.001167648999398807, 0.0009137039996858221, 0.0013712540003325557, 0.0016304350001519197, 0.0009206579998135567]


In [ ]:
dist_bellman_ford = []
bellman_ford_times = []
for i in range(10):
  t2 = time.perf_counter()
  dist_bellman_ford.append(bellman_ford(dot, adj_list))
  t3 = time.perf_counter()
  bellman_ford_times.append(t3 - t2)
print(bellman_ford_times)

[0.017364500999974553, 0.015762125000037486, 0.013176853000004485, 0.01316944300015166, 0.012845689000641869, 0.013774580999779573, 0.011277802000222437, 0.0069949670005371445, 0.0069424450002770755, 0.006917952000549121]


In [ ]:
print(dist_dijkstra[0] == dist_bellman_ford[0])
print(dist_dijkstra[0][:10])

True
[11, 7, 8, 5, 6, 9, 5, 7, 7, 8]


In [ ]:
print(f"Average time for Dijkstra: {(sum(dijkstra_times) / len(dijkstra_times)):.6f}\n"
f"Minimal time for Dijkstra: {min(dijkstra_times):.6f}\n"
f"Maximal time for Dijkstra: {max(dijkstra_times):.6f}\n\n"
f"Average time for Bellman-Ford: {(sum(bellman_ford_times) / len(bellman_ford_times)):.6f}\n"
f"Minimal time for Bellman-Ford: {min(bellman_ford_times):.6f}\n"
f"Maximal time for Bellman-Ford: {max(bellman_ford_times):.6f}"
)

Average time for Dijkstra: 0.001418
Minimal time for Dijkstra: 0.000914
Maximal time for Dijkstra: 0.002649

Average time for Bellman-Ford: 0.011823
Minimal time for Bellman-Ford: 0.006918
Maximal time for Bellman-Ford: 0.017365


#### Task 1 conclusion  
Both methods found the same paths; however, the average execution time for Dijkstra's method was significantly shorter, which is consistent with the algorithms' respective complexities: O(|V|²) for Dijkstra's and O(|V|·|E|) for Bellman-Ford.

## Task 2

- Generate a 10x20 cell grid with 40 obstacle cells. Choose two random non
obstacle cells and find a shortest path between them using A* algorithm. Repeat the
experiment 5 times with different random pair of cells. Analyse the results obtained.   

In [ ]:
def generate_grid():
  grid = [[0] * 20 for _ in range(10)]
  obstacles = 0
  while obstacles < 40:
    i = random.randrange(10)
    j = random.randrange(20)
    if grid[i][j] == 0:
      grid[i][j] = 1
      obstacles += 1
  return grid

In [ ]:
def neighbours(i, j, grid):
  directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
  neibs = []
  for dx, dy in directions:
    if 0 <= i + dx <= 9 and 0 <= j + dy <= 19:
      if grid[i + dx][j + dy] == 0:
        neibs.append((i + dx, j + dy))
  return neibs

In [ ]:
def manhattan(a, b):
  r1, c1 = a
  r2, c2 = b
  return abs(r2 - r1) + abs(c2 - c1)

In [ ]:
def a(start, end, grid):
  open_set = [start]
  closed = []
  g = {start : 0}
  parents = {start : None}
  while open_set:
    best = float("inf")
    c = None
    for i in open_set:
      f = g[i] + manhattan(i, end)
      if f < best:
        best = f
        c = i
    if c == end:
      break
    open_set.remove(c)
    closed.append(c)
    for n in neighbours(c[0], c[1], grid):
      if n in closed:
        continue
      new_g = g[c] + 1
      if n not in g or new_g < g[n]:
        g[n] = new_g
        parents[n] = c
        if n not in open_set:
          open_set.append(n)
  if end not in g:
    return None
  path = []
  c = end
  while c != start:
    path.append(c)
    c = parents[c]
  path.append(start)
  return path[::-1]

In [ ]:
random.seed(300)
grid = generate_grid()
free = [(i, j) for i in range(10) for j in range(20) if grid[i][j] == 0]
for _ in range(5):
  pair = random.sample(free, 2)
  path = a(pair[0], pair[1], grid)
  if path is None:
    print(pair[0], pair[1], "No path")
  else:
    print(pair[0], pair[1], "length:", len(path) - 1, "manhattan:", manhattan(pair[0], pair[1]))

(9, 8) (9, 0) length: 12 manhattan: 8
(6, 3) (1, 7) length: 9 manhattan: 9
(7, 6) (7, 10) length: 4 manhattan: 4
(7, 18) (5, 18) length: 2 manhattan: 2
(0, 4) (4, 10) length: 10 manhattan: 10


#### Task 2 conclusion  
A* found the shortest paths for all pairs; there were no instances where no path existed. In 4 out of 5 cases, the shortest path was equal in length to the path that would exist if there were no obstacles. In one case, the Manhattan distance was 8, while the actual distance was 12—this was because the path encountered obstacles.

#### Data structures and design techniques

The following data structures were used in this work:
- an adjacency matrix and an adjacency list (storing neighbors and weights)
- lists used as arrays
- dictionaries (for graph representation and parent tracking)
- a two-dimensional list used as a grid
And the following methods:
- relaxation (updating a value if a shorter distance is found)
- heuristics (estimating the distance, assuming no obstacles, for the A* algorithm)